# Florence-2 Hasar Sınıflandırması - Google Colab Eğitim Notebook'u

Bu notebook, **A100 GPU** kullanarak Florence-2-base modelini kendi kırpılmış hasar görsellerinizle (Çizik, Göçük vb.) VQA / Sınıflandırma formatında **LoRA (PEFT)** ile eğitmek amacıyla hazırlanmıştır.

## Kullanım Adımları
1. Bu `.ipynb` notebook dosyasını Drive'ınızdaki `model-hasar-egitim` klasörüne yükleyip Colab ile açın.
2. `Runtime > Change runtime type > A100 GPU` seçin.
3. Hücreleri sırayla çalıştırın.
4. Eğitilen `florence_hades_lora.zip` dosyasını indirip projenize dahil edin.

---
### 1. GPU Kontrolü ve Bağımlılıkların Kurulumu

In [ ]:
#@title GPU Kontrolü
import torch
print(f"CUDA mevcut: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("GPU bulunamadı! Runtime > Change runtime type > A100 GPU seçin.")

In [ ]:
#@title Kütüphanelerin Kurulumu
!pip install -q transformers peft bitsandbytes accelerate datasets
import transformers, peft, accelerate
print("Kurulum tamamlandı.")
print(f"Transformers: {transformers.__version__}")

---
### 2. Google Drive Bağlama ve Veri Seti Açma

Drive'ınızdaki `model-hasar-egitim/hasar-ornek-labelli.zip` dosyasını Colab hızlı diskine ziple açıyoruz.

In [ ]:
#@title Google Drive Bağlama ve ZIP Çıkarma
from google.colab import drive
import zipfile
import os
from pathlib import Path

drive.mount('/content/drive')

# Drive'daki zip konumu
ZIP_YOLU = Path("/content/drive/MyDrive/model-hasar-egitim/hasar-ornek-labelli.zip")
HEDEF_DIZIN = Path("/content/hasar_data")

if ZIP_YOLU.exists():
    print(f"ZIP dosyasi bulundu: {ZIP_YOLU}")
    with zipfile.ZipFile(ZIP_YOLU, 'r') as zip_ref:
        zip_ref.extractall(HEDEF_DIZIN)
    print(f"Zip basariyla cikarildi: {HEDEF_DIZIN}")
else:
    print(f"[!] UYARI: Zip bulunamadi -> {ZIP_YOLU}")
    print("Lutfen Drive'da 'model-hasar-egitim/hasar-ornek-labelli.zip' dosyasinin bulundugundan emin olun.")

In [ ]:
#@title Veri Setini Ayrıştırma ve Kırpma İndeksi Oluşturma
import random
from PIL import Image
from torch.utils.data import Dataset

SINIF_ISIMLERI = {
    0: "Cizik",
    1: "Gocuk",
    2: "Cam Kirigi",
    3: "Pas",
    4: "Kus Pisligi"
}

def verileri_tara(data_dir):
    veri = []
    # Zip tek klasore mi cikti yoksa direkt icinde mi kontrol et
    tum_gorseller = list(Path(data_dir).rglob("*.jpg")) + list(Path(data_dir).rglob("*.png"))
    
    for img_path in tum_gorseller:
        lbl_path = img_path.with_suffix(".txt")
        if not lbl_path.exists(): continue
        
        with open(lbl_path, 'r', encoding='utf-8') as f:
            satirlar = f.readlines()
            
        for satir in satirlar:
            parcalar = satir.strip().split()
            if len(parcalar) >= 5:
                sinif_id = int(parcalar[0])
                sinif_adi = SINIF_ISIMLERI.get(sinif_id, f"Sinif_{sinif_id}")
                x_center, y_center, w, h = map(float, parcalar[1:5])
                veri.append({
                    "image_path": str(img_path),
                    "class_name": sinif_adi,
                    "bbox": [x_center, y_center, w, h]
                })
    return veri

tum_veri = verileri_tara(HEDEF_DIZIN)
random.seed(42)
random.shuffle(tum_veri)

split_idx = int(len(tum_veri) * 0.85)
train_data = tum_veri[:split_idx]
val_data = tum_veri[split_idx:]

print(f"Toplam Kirpilacak Hasar Nesnesi: {len(tum_veri)}")
print(f"Egitim Parcalari: {len(train_data)} | Dogrulama Parcalari: {len(val_data)}")

In [ ]:
#@title PyTorch Dataset Tanımı
class FlorenceHasarDataset(Dataset):
    def __init__(self, veri_listesi, processor):
        self.veri = veri_listesi
        self.processor = processor
        self.prompt = "<DETAILED_CAPTION>"
        
    def __len__(self):
        return len(self.veri)
        
    def __getitem__(self, idx):
        item = self.veri[idx]
        img_path = item["image_path"]
        class_name = item["class_name"]
        x_c, y_c, w_n, h_n = item["bbox"]
        
        img = Image.open(img_path).convert("RGB")
        img_w, img_h = img.size
        
        x_center = x_c * img_w
        y_center = y_c * img_h
        width = w_n * img_w
        height = h_n * img_h
        
        x1 = max(0, int(x_center - width / 2))
        y1 = max(0, int(y_center - height / 2))
        x2 = min(img_w, int(x_center + width / 2))
        y2 = min(img_h, int(y_center + height / 2))
        
        # Nesne bolgesini kirp
        crop_img = img.crop((x1, y1, x2, y2))
        
        inputs = self.processor(text=self.prompt, images=crop_img, return_tensors="pt", padding="max_length", max_length=128, truncation=True)
        labels = self.processor.tokenizer(text=class_name, return_tensors="pt", padding="max_length", max_length=16, truncation=True).input_ids
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        
        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0),
            "pixel_values": inputs["pixel_values"].squeeze(0),
            "labels": labels.squeeze(0)
        }


---
### 3. Model Yükleme ve LoRA Adaptörü


In [ ]:
#@title Florence-2-base ve LoRA Hazırlığı
from transformers import AutoModelForCausalLM, AutoProcessor
from peft import get_peft_model, LoraConfig
import torch

MODEL_ID = "microsoft/Florence-2-base"

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, 
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

train_dataset = FlorenceHasarDataset(train_data, processor)
val_dataset = FlorenceHasarDataset(val_data, processor)

---
### 4. Model Eğitimi (Fine-Tuning)


In [ ]:
#@title Eğitimi Başlat
from transformers import TrainingArguments, Trainer

Cikti_Dizini = "/content/florence_hades_output"

training_args = TrainingArguments(
    output_dir=Cikti_Dizini,
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    fp16=False,
    bf16=True, # A100 GPU bf16 destekler
    logging_steps=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("Egitim Basliyor...")
trainer.train()

---
### 5. Modeli Kaydetme ve İndirme


In [ ]:
#@title Ağırlıkları Drive'a ve Bilgisayara İndirme
import shutil
from google.colab import files

Kayit_Dizini = "/content/florence_finetuned"
model.save_pretrained(Kayit_Dizini)
processor.save_pretrained(Kayit_Dizini)

# Drive'a da yedekle
DRIVE_KAYIT = Path("/content/drive/MyDrive/model-hasar-egitim/florence_finetuned")
model.save_pretrained(DRIVE_KAYIT)
processor.save_pretrained(DRIVE_KAYIT)
print(f"Drive'a yedeklendi: {DRIVE_KAYIT}")

# Zip'le ve bilgisayara indir
zip_yolu = "/content/florence_hades_lora.zip"
shutil.make_archive(zip_yolu.replace('.zip',''), 'zip', Kayit_Dizini)

print(f"Kaydedildi: {zip_yolu}")
files.download(zip_yolu)